In [1]:
from dataclasses import fields

import pandas as pd
from IPython.display import display

import gem
from gem.constants import hero_display

# added a convenience function to download the dem file from api and decompress it
# `fetch_replay` wraps `fetch_replay_url` and `download_and_decompress`
dem_path = gem.fetch_replay(8734577999, out_dir="replays/")
match = gem.parse(str(dem_path))

# alternatively, a 2-step appraoch

# url = gem.fetch_replay_url(8734577999)
# dem_path = gem.download_and_decompress(8734577999, url, out_dir="replays/")
# match = gem.parse(str(dem_path))

In [2]:
# a snapshot of the fields
[f.name for f in fields(match)][:15]

['match_id',
 'game_mode',
 'leagueid',
 'radiant_win',
 'radiant_team_id',
 'radiant_team_name',
 'radiant_team_tag',
 'dire_team_id',
 'dire_team_name',
 'dire_team_tag',
 'game_start_tick',
 'game_end_tick',
 'players',
 'towers',
 'barracks']

In [3]:
# Team identity (available for league/tournament games)
print(f"Radiant: {match.radiant_team_name} [{match.radiant_team_tag}] (id={match.radiant_team_id})")
print(f"Dire:    {match.dire_team_name} [{match.dire_team_tag}] (id={match.dire_team_id})")
print()

# Build draft DataFrame
sorted_draft = sorted(match.draft, key=lambda d: d.tick)
hero_to_player = {p.hero_name: p for p in match.players}


rows = []
for i, ev in enumerate(sorted_draft, 1):
    pp = hero_to_player.get(ev.hero_name)
    team = gem.resolve_pick_team(ev, match.players)
    rows.append(
        {
            "#": i,
            "type": "PICK" if ev.is_pick else "BAN",
            "team": "Radiant" if team == 2 else "Dire",
            "hero": hero_display(ev.hero_name) if ev.hero_name else f"ID {ev.hero_id}",
            "player": pp.player_name if pp else "",
            "account_id": pp.account_id if pp else "",
            "tick": ev.tick,
        }
    )
df = pd.DataFrame(rows)


# Colour rows
def _style(row):
    colors = {
        ("PICK", "Radiant"): "background-color: #1a3a1a; color: #7fba7f",
        ("PICK", "Dire"): "background-color: #1a1a3a; color: #7f7fba",
        ("BAN", "Radiant"): "background-color: #2a1a1a; color: #ba7f7f",
        ("BAN", "Dire"): "background-color: #2a1a2a; color: #ba7fba",
    }
    color = colors.get((row["type"], row["team"]), "")
    return [color] * len(row)


display(df.style.apply(_style, axis=1).set_caption("Draft sequence").hide(axis="index"))

# Picks by team summary — now includes account_id for cross-referencing with OpenDota/Dotabuff
print(f"\nRadiant picks ({match.radiant_team_name}):")
display(
    df[(df.type == "PICK") & (df.team == "Radiant")][
        ["#", "hero", "player", "account_id", "tick"]
    ].reset_index(drop=True)
)

print(f"\nDire picks ({match.dire_team_name}):")
display(
    df[(df.type == "PICK") & (df.team == "Dire")][
        ["#", "hero", "player", "account_id", "tick"]
    ].reset_index(drop=True)
)

Radiant: Team Spherical [spherical] (id=9156258)
Dire:    Vy Guys [Vy-Guys] (id=10040107)



#,type,team,hero,player,account_id,tick
1,BAN,Dire,Magnus,,,187
2,BAN,Radiant,Mars,,,495
3,BAN,Radiant,Axe,,,955
4,BAN,Dire,Oracle,,,1059
5,BAN,Radiant,Kunkka,,,1081
6,BAN,Radiant,Lina,,,1531
7,BAN,Dire,Viper,,,1687
8,PICK,Dire,Snapfire,xXPhilosophyEnjoyerXx,54551999,1861
9,PICK,Radiant,Spirit Breaker,Moist Goblin,124961459,2125
10,BAN,Dire,Legion Commander,,,2523



Radiant picks (Team Spherical):


,#,hero,player,account_id,tick
0,9,Spirit Breaker,Moist Goblin,124961459,2125
1,13,Bane,Spartomat,151610348,3959
2,16,Pangolier,o в ѕ т,397510025,7547
3,17,Faceless Void,Larry Llama,198617104,8945
4,24,Shadow Fiend,sleepymoon,288762294,15479



Dire picks (Vy Guys):


,#,hero,player,account_id,tick
0,8,Snapfire,xXPhilosophyEnjoyerXx,54551999,1861
1,14,Vengeful Spirit,Vy,1118150174,4625
2,15,Dawnbreaker,Maselway,85085356,5973
3,18,Windranger,Rami,40283487,9685
4,23,Bloodseeker,Lunst Valentine,50155876,14363
